In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib import cm
from matplotlib.colors import Normalize
from collections import defaultdict

import yaml
import h5py
import tqdm
import json
# from larndsim.fee import digitize


In [ ]:
def unique_channel_id(d):
    return ((d['io_group'].astype(int)*10000+d['io_channel'].astype(int))*1000 \
            + d['chip_id'].astype(int))*100 + d['channel_id'].astype(int)

def unique_to_channel_id(unique):
    return unique % 100

def unique_to_chip_id(unique):
    return (unique// 100) % 1000

def unique_to_io_channel(unique):
    return(unique//(100*1000)) % 1000

def unique_to_tiles(unique):
    return ( (unique_to_io_channel(unique)-1) // 4) + 1

def unique_to_io_group(unique):
    return(unique // (100*1000*10000)) % 10000



In [ ]:
# _default_geometry_yaml = '../larndsim/pixel_layouts/multi_tile_layout-2.4.16_v4.yaml'
# _default_geometry_yaml_mod2 = '../larndsim/pixel_layouts/multi_tile_layout-2.5.16_v4.yaml'

def _default_pxy():
    return (0., 0.)


def _rotate_pixel(pixel_pos, tile_orientation):
    return pixel_pos[0]*tile_orientation[2], pixel_pos[1]*tile_orientation[1]


cmap = cm.viridis_r
# pixel_pitch = 1

class Geo:
    def __init__(self, geometry_yaml):
        with open(geometry_yaml) as fi:
            geo = yaml.full_load(fi)
        
        self.pixel_pitch = geo['pixel_pitch']
        
        chip_channel_to_position = geo['chip_channel_to_position']
        tile_orientations = geo['tile_orientations']
        tile_positions = geo['tile_positions']
        tpc_centers = geo['tpc_centers']
        tile_indeces = geo['tile_indeces']
        xs = np.array(list(chip_channel_to_position.values()))[
            :, 0] * self.pixel_pitch
        ys = np.array(list(chip_channel_to_position.values()))[
            :, 1] * self.pixel_pitch
        x_size = max(xs)-min(xs)+self.pixel_pitch
        y_size = max(ys)-min(ys)+self.pixel_pitch
        
        tile_geometry = defaultdict(int)
        io_group_io_channel_to_tile = {}
        self.geometry = defaultdict(_default_pxy)
        
        for tile in geo['tile_chip_to_io']:
            tile_orientation = tile_orientations[tile]
            tile_geometry[tile] = tile_positions[tile], tile_orientations[tile]
            for chip in geo['tile_chip_to_io'][tile]:
                io_group_io_channel = geo['tile_chip_to_io'][tile][chip]
                io_group = io_group_io_channel//1000
                io_channel = io_group_io_channel % 1000
                io_group_io_channel_to_tile[(
                    io_group, io_channel)] = tile
        
            for chip_channel in geo['chip_channel_to_position']:
                chip = chip_channel // 1000
                channel = chip_channel % 1000
                try:
                    io_group_io_channel = geo['tile_chip_to_io'][tile][chip]
                except KeyError:
                    print("Chip %i on tile %i not present in network" %
                          (chip, tile))
                    continue
        
                io_group = io_group_io_channel // 1000
                io_channel = io_group_io_channel % 1000
                x = chip_channel_to_position[chip_channel][0] * \
                    self.pixel_pitch + self.pixel_pitch / 2 - x_size / 2
                y = chip_channel_to_position[chip_channel][1] * \
                    self.pixel_pitch + self.pixel_pitch / 2 - y_size / 2
        
                x, y = _rotate_pixel((x, y), tile_orientation)
                x += tile_positions[tile][2] + \
                    tpc_centers[tile_indeces[tile][0]][0]
                y += tile_positions[tile][1] + \
                    tpc_centers[tile_indeces[tile][0]][1]
        
                self.geometry[(io_group, io_group_io_channel_to_tile[(
                    io_group, io_channel)], chip, channel)] = x, y
        
        self.xmin = min(np.array(list(self.geometry.values()))[:, 0])-self.pixel_pitch/2
        self.xmax = max(np.array(list(self.geometry.values()))[:, 0])+self.pixel_pitch/2
        self.ymin = min(np.array(list(self.geometry.values()))[:, 1])-self.pixel_pitch/2
        self.ymax = max(np.array(list(self.geometry.values()))[:, 1])+self.pixel_pitch/2
        
        tile_vertical_lines = np.linspace(self.xmin, self.xmax, 3)
        tile_horizontal_lines = np.linspace(self.ymin, self.ymax, 5)
        chip_vertical_lines = np.linspace(self.xmin, self.xmax, 21)
        chip_horizontal_lines = np.linspace(self.ymin, self.ymax, 41)


In [ ]:
def unique_id_to_pixel_id(un, geometry_object):
    io_group = unique_to_io_group(un)
    tile = unique_to_io_channel(un)
    chip_id = unique_to_chip_id(un)
    channel_id = unique_to_channel_id(un)

    pitch = geometry_object.pixel_pitch

    gg = geometry_object.geometry

    x, y = gg[(2 - (io_group % 2), tile + 8 * (1 - (io_group % 2)), chip_id, channel_id)]

    x_min = geometry_object.xmin
    x_max = geometry_object.xmax
    
    y_min = geometry_object.ymin
    y_max = geometry_object.ymax

    x_int = (x - x_min) / pitch - 0.5
    y_int = (y - y_min) / pitch - 0.5

    if abs(round(x_int) - x_int) > 0.05:
        # print(is_mod2)
        print('ERROR X: ', un, ' - ', round(x_int), ' vs ', x_int)
    if abs(round(y_int) - y_int) > 0.05:
        # print(is_mod2)
        print('ERROR Y: ', un, ' - ', round(y_int), ' vs ', y_int)
        
    if abs(round((x_max - x_min)/pitch) - (x_max - x_min)/pitch) > 0.05:
        print('ERROR STEP X: ', un)
    if abs(round((y_max - y_min)/pitch) - (y_max - y_min)/pitch) > 0.05:
        print('ERROR STEP Y: ', un)

    npix_x = round((x_max - x_min)/pitch)
    npix_y = round((y_max - y_min)/pitch)
    
    return round(x_int) + (round(y_int) + npix_y * (1 - (io_group % 2))) * npix_x
    

In [ ]:
# LArPix-v2a anodes
nonrouted_v2a_channels = [6, 7, 8, 9, 22, 23, 24, 25, 38, 39, 40, 54, 55, 56, 57]


geo_object_mod0 = Geo("../larndsim/pixel_layouts/multi_tile_layout-2.3.16_mod0_swap_T8T4T7.yaml")
geo_object_mod1 = Geo("../larndsim/pixel_layouts/multi_tile_layout-2.3.16_mod1_noswap.yaml")
geo_object_mod3 = Geo("../larndsim/pixel_layouts/multi_tile_layout-2.3.16_mod3_swap_T5T8_T9T10.yaml")

for i_mod, geo_object in zip([0,1,3], [geo_object_mod0, geo_object_mod1, geo_object_mod3]):
    pixelid_to_uniqueid = dict()
    uniqueid_to_pixelid = dict()

    for io_group in tqdm.tqdm(range(1, 3)):
        for tile in range(1, 9):
            for chip_id in range(11, 111):
                for channel_id in range(64):
                    if io_group in [1, 2, 3, 4, 7, 8] and channel_id in nonrouted_v2a_channels:
                        continue
                    unique_id = ((io_group*10000+tile)*1000 + chip_id)*100 + channel_id
                    pixel_id = unique_id_to_pixel_id(unique_id, geo_object)
    
                    if pixel_id in pixelid_to_uniqueid.keys():
                        print('DUPLICATE PIXEL ID')
                        print('pixel_id: ', pixel_id)
                        print('channel: ', (io_group, tile, chip_id, channel_id))
                    if channel_id in uniqueid_to_pixelid.keys():
                        print('DUPLICATE UNIQUE ID')
                    pixelid_to_uniqueid[pixel_id] = unique_id
                    uniqueid_to_pixelid[unique_id] = pixel_id
    
    with open("pixelid_to_uniqueid_mod{}.json".format(i_mod), "w") as f:
        json.dump(pixelid_to_uniqueid, f)
    with open("uniqueid_to_pixelid_mod{}.json".format(i_mod), "w") as f:
        json.dump(uniqueid_to_pixelid, f)


In [ ]:
# LArPix-v2b anodes

pixelid_to_uniqueid = dict()
uniqueid_to_pixelid = dict()

geo_object_mod2 = Geo("../larndsim/pixel_layouts/multi_tile_layout-2.5.16_mod2_swap_T7T8.yaml")


for io_group in tqdm.tqdm(range(1, 3)):
    for tile in range(1, 9):
        for chip_id in range(11, 111):
            for channel_id in range(64):
                unique_id = ((io_group*10000+tile)*1000 + chip_id)*100 + channel_id
                pixel_id = unique_id_to_pixel_id(unique_id, geo_object_mod2)

                if pixel_id in pixelid_to_uniqueid.keys():
                    print('DUPLICATE PIXEL ID')
                    print('pixel_id: ', pixel_id)
                    print('channel: ', (io_group, tile, chip_id, channel_id))
                if channel_id in uniqueid_to_pixelid.keys():
                    print('DUPLICATE UNIQUE ID')
                pixelid_to_uniqueid[pixel_id] = unique_id
                uniqueid_to_pixelid[unique_id] = pixel_id

with open("pixelid_to_uniqueid_mod2.json", "w") as f:
    json.dump(pixelid_to_uniqueid, f)
with open("uniqueid_to_pixelid_mod2.json", "w") as f:
    json.dump(uniqueid_to_pixelid, f)
